# Projek Hybrid GraphRAG untuk Permainan Kartu Yu-Gi-Oh!

Notebook eksperimen ini mendemonstrasikan arsitektur end-to-end **Hybrid GraphRAG** menggunakan **Neo4j Knowledge Graph + Vector Index + Fulltext BM25** dan **Google Gemini API (`google-genai`)** di atas 4 sumber dokumen utama:
1. **Official Rulebook v10 + Master Rule Revision (2020/2021 Update)** — aturan dasar & zona Extra Deck terbaru.
2. **Database Informasi Kartu (~14.000+ kartu resmi)** dari YGOPRODeck API v7.
3. **Problem-Solving Card Text (PSCT) & Official Card Rulings/FAQ** dari Konami & `db.ygoresources.com`.
4. **Daftar Kartu Forbidden & Limited (Banlist)** untuk format TCG, OCG, dan Goat.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import get_settings

settings = get_settings()
print("Project Root :", PROJECT_ROOT)
print("Neo4j URI    :", settings.neo4j_uri)
print("Gemini Model :", settings.gemini_llm_model, "| Embedding:", settings.gemini_embedding_model)

## 1. Akuisisi Otomatis Keempat Sumber Dokumen & Analisis PSCT

Menjalankan pipeline pengunduhan otomatis (menggunakan cache lokal jika sudah terunduh sebelumnya).

In [ ]:
from src.ingestion.fetch_cards_banlist import process_cards_and_banlist
from src.ingestion.fetch_rulebook import process_rulebook
from src.ingestion.fetch_psct_rulings import process_psct_and_rulings

cards_enriched, banlist_entries = process_cards_and_banlist(settings)
rulebook_chunks = process_rulebook(settings)
psct_rules, rulings_bundle = process_psct_and_rulings(
    cards_enriched=cards_enriched,
    settings=settings,
    max_priority_cards=250,
    max_qa_entries=600,
)

print(f"1. Rulebook & Master Rule Chunks : {len(rulebook_chunks):,}")
print(f"2. Total Kartu Yu-Gi-Oh!         : {len(cards_enriched):,}")
print(f"3. Aturan PSCT                   : {len(psct_rules):,}")
print(f"   Official Card FAQs            : {len(rulings_bundle['faqs']):,}")
print(f"   Official Q&A Rulings          : {len(rulings_bundle['rulings']):,}")
print(f"4. Total Entri Banlist           : {len(banlist_entries):,}")

## 2. Exploratory Data Analysis (EDA) & Bedah Tata Bahasa PSCT

Mari kita lihat distribusi tipe kartu, perbandingan Banlist TCG vs OCG, dan hasil bedah PSCT otomatis pada kartu kompetitif seperti **Ash Blossom & Joyous Spring** dan **Mirrorjade the Iceblade Dragon**.

In [ ]:
df_cards = pd.DataFrame(cards_enriched)
print("=== 10 Tipe Kartu Terbanyak ===")
display(df_cards["type"].value_counts().head(10).to_frame("jumlah_kartu"))

df_ban = pd.DataFrame(banlist_entries)
print("\n=== Distribusi Banlist per Format ===")
display(pd.crosstab(df_ban["format"], df_ban["status"]))

In [ ]:
# Contoh hasil pembedahan PSCT otomatis
for sample_name in ["Ash Blossom & Joyous Spring", "Mirrorjade the Iceblade Dragon"]:
    sample = next((c for c in cards_enriched if c["name"] == sample_name), None)
    if sample:
        print(f"=== {sample['name']} (TCG: {sample['ban_tcg']}, OCG: {sample['ban_ocg']}) ===")
        print("PSCT Rules Terdeteksi :", sample["psct"]["psct_rule_ids"])
        print("Targeting Aktivasi    :", sample["psct"]["has_targeting"], "| Memiliki Cost (;):", sample["psct"]["has_cost"])
        for i, cl in enumerate(sample["psct"]["clauses"], 1):
            print(f"  Klausa {i} -> Kondisi (:): {cl['condition']!r} | Cost/Target (;): {cl['activation_and_cost']!r}")
        print()

## 3. Konstruksi Knowledge Graph & Vector Index di Neo4j

Pastikan container Neo4j sudah menyala (`docker compose up -d`), kemudian jalankan sel berikut untuk membuat skema constraint, indeks fulltext BM25, indeks vektor 768 dimensi, dan mengingesti keempat dokumen.

In [ ]:
from src.graph.neo4j_client import Neo4jGraphClient
from src.graph.schema_and_ingest import (
    setup_neo4j_schema,
    ingest_rulebook_to_neo4j,
    ingest_psct_rules_to_neo4j,
    ingest_cards_and_banlist_to_neo4j,
    ingest_rulings_to_neo4j,
    embed_and_index_nodes,
)
from src.graph.embedder import GeminiEmbedder

neo_client = Neo4jGraphClient(settings)
connected, status_msg = neo_client.check_connection()
print("Status Neo4j:", status_msg)

if connected:
    setup_neo4j_schema(neo_client, embedding_dim=settings.embedding_dimension)
    ingest_rulebook_to_neo4j(neo_client, rulebook_chunks)
    ingest_psct_rules_to_neo4j(neo_client, psct_rules)
    ingest_cards_and_banlist_to_neo4j(neo_client, cards_enriched, banlist_entries)
    ingest_rulings_to_neo4j(neo_client, rulings_bundle)

    if settings.gemini_api_key:
        embedder = GeminiEmbedder(settings)
        embed_and_index_nodes(
            client=neo_client,
            embedder=embedder,
            rule_chunks=rulebook_chunks,
            psct_rules=psct_rules,
            rulings_bundle=rulings_bundle,
            cards_enriched=cards_enriched,
            max_card_embeddings=600,  # Ubah ke None untuk embed seluruh ~14.000 kartu
        )
    display(pd.Series(neo_client.get_graph_statistics(), name="jumlah").to_frame())

## 4. Eksperimen Mesin Hybrid GraphRAG pada 4 Skenario Uji

Menguji integrasi **Query Router -> Multi-Hop Graph Retriever -> Gemini Head Judge Generator** untuk keempat domain dokumen.

In [ ]:
from IPython.display import Markdown, display
from src.rag.router import QueryRouter
from src.rag.retriever import HybridGraphRetriever
from src.rag.generator import GraphRAGGenerator

router = QueryRouter(settings)
retriever = HybridGraphRetriever(settings, neo4j_client=neo_client)

test_questions = [
    # 1. Official Rulebook + Master Rule
    "Bagaimana aturan Summon Synchro dan Link Monster dari Extra Deck ke Main Monster Zone di Master Rule terbaru?",
    # 2. Informasi Kartu + Relasi Archetype
    "Jelaskan efek kartu Fallen of Albaz dan kartu-kartu apa saja yang berhubungan dengannya!",
    # 3. Card Ruling + PSCT
    "Apakah Ash Blossom & Joyous Spring bisa men-negate efek saat Damage Step? Jelaskan berdasarkan PSCT dan aturan Damage Step!",
    # 4. Banlist
    "Apa perbedaan status banlist Maxx \"C\" dan Baronne de Fleur di TCG dibanding OCG?",
]

for idx, q in enumerate(test_questions, 1):
    plan = router.route(q, use_llm_router=bool(settings.gemini_api_key))
    ret = retriever.retrieve(plan, top_k=5)
    print(f"\n{'=' * 80}\nSkenario {idx}: {q}")
    print(f"Mode: {ret.mode} | Intents: {plan.intents} | Graph Paths: {len(ret.graph_paths)}")
    for p in ret.graph_paths[:5]:
        print("  ->", p)
    if settings.gemini_api_key:
        gen = GraphRAGGenerator(settings)
        ans = gen.generate(q, ret)
        display(Markdown(ans))
    else:
        print("(Isi GEMINI_API_KEY di .env untuk menampilkan jawaban LLM Head Judge)")

## 5. Evaluasi Kuantitatif 7 Skenario Wajib & RAGAS (Bahan BAB 4 Laporan)

Menampilkan hasil pengukuran dari `results/experiment_results.json` (jalankan `python scripts/run_experiments.py` untuk memperbarui angka eksperimen).

In [ ]:
import json
res_path = PROJECT_ROOT / "results" / "experiment_results.json"
if res_path.exists():
    bench = json.loads(res_path.read_text(encoding="utf-8"))
    print("=== 1. Vector vs. Graph vs. Hybrid (Recall@K & MRR) ===")
    display(pd.DataFrame(bench["exp1_vector_vs_graph_vs_hybrid"]))
    print("\n=== 2. Single-hop vs. Multi-hop Queries (Answer Accuracy) ===")
    display(pd.DataFrame(bench["exp2_single_vs_multihop"]))
    print("\n=== 3. Graph Traversal Query Latency (1/2/3-Hop) ===")
    display(pd.DataFrame(bench["exp3_graph_traversal_latency"]))
    print("\n=== 4. Integrasi Graph + Vector (End-to-End Retrieval Latency) ===")
    display(pd.DataFrame(bench["exp4_e2e_integration_latency"]))
    print("\n=== 5. Pertumbuhan Node & Edge (Latency & Storage Size) ===")
    display(pd.DataFrame(bench["exp5_node_edge_growth"]))
    print("\n=== 6. Entity & Relationship Extraction (Precision, Recall, F1) ===")
    display(pd.DataFrame(bench["exp6_entity_relationship_extraction"]["summary"]))
    print("\n=== 7. Evaluasi Generatif LLM dengan RAGAS ===")
    print("Agregat RAGAS:", bench["exp7_ragas_evaluation"]["aggregate"])
    display(pd.DataFrame(bench["exp7_ragas_evaluation"]["samples"]))
else:
    print("Jalankan `python scripts/run_experiments.py` terlebih dahulu.")